# <center>Filtering Youtube Videos</center>

*Usa a API para extrair informações dos vídeos do yt e guarda em youtube_titulos_saida.csv*

---

In [1]:
!pip install google-api-python-client

  Using cached httplib2-0.31.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached google_auth-2.40.3-py2.py3-none-any.whl.metadata (6.2 kB)
  Using cached google_auth_httplib2-0.2.0-py2.py3-none-any.whl.metadata (2.2 kB)
  Using cached google_api_core-2.25.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached uritemplate-4.2.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached googleapis_common_protos-1.70.0-py3-none-any.whl.metadata (9.3 kB)
  Using cached proto_plus-1.26.1-py3-none-any.whl.metadata (2.2 kB)
  Using cached cachetools-5.5.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached rsa-4.9.1-py3-none-any.whl.metadata (5.6 kB)
  Using cached pyasn1-0.6.1-py3-none-any.whl.metadata (8.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 42.9 MB/s  0:00:006m0:00:01
Using cached google_api_core-2.25.1-py3-none-any.whl (160 kB)
Using cached google_auth-2.40.3-py2.py3-none-any.whl (216 kB)
Using cached cachet

In [8]:
import os
import re
import json

import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

## Analisando quantidade de cada conteúdo

In [13]:
file_path = "data/youtube_links.jsonl"

In [3]:
from urllib.parse import urlparse

def rate_link(url: str) -> str:
    url = url.strip().lower()
    if not url.startswith("http"):
        return "outro"

    try:
        parsed = urlparse(url)
        path = parsed.path
    except Exception:
        return "outro"

    # --- Lives ---
    if "/live/" in path or "feature=live" in url or "&live=1" in url:
        return "live"
    
    # --- Shorts ---
    if "/shorts/" in path:
        return "short"
    
    # --- Vídeos ---
    if "youtu.be/" in url or "/watch" in path:
        return "video"

    return "outro"

In [4]:
import gzip
import json
import re
from tqdm import tqdm
from urllib.parse import urlparse, parse_qs

# Definições dos tipos de conteúdo
TIPOS = {
    "video": 0,
    "short": 0,
    "live": 0,
    "outro": 0,
}

arquivo = "youtube_links.jsonl"   

with open(arquivo, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Classificando links"):
        try:
            data = json.loads(line)
            url = data.get("url")
            if not url:
                continue

            tipo = rate_link(url)
            TIPOS[tipo] += 1

        except json.JSONDecodeError:
            continue

# ======================
# RESULTADO FINAL
# ======================
print("\n=== Quantidade de cada tipo de link no YouTube ===")
for tipo, qtd in TIPOS.items():
    print(f"{tipo}: {qtd}")

print(f"\nTOTAL: {sum(TIPOS.values())}")

Classificando links: 1920792it [00:50, 37942.47it/s]


=== Quantidade de cada tipo de link no YouTube ===
video: 1602380
short: 161251
live: 137615
outro: 19546

TOTAL: 1920792


### Extraindo informações dos links
##### ID | título | descrição | canal | data de publicação | link | visualizações | likes| comentários(qtd)

In [9]:
output_csv_path = "data/youtube_titulos_saida.csv"

In [10]:
def get_ID(url:str) -> str:
    #match = re.search(
    #        r"(?:v=|youtu\.be/|/v/|/embed/|/live/|/shorts/)([a-zA-Z0-9_-]{11})",
    #        url
    #    )
    match = re.search(
                r'(?:v=|\/|be\/|embed\/|shorts\/|watch\?v=)([a-zA-Z0-9_-]{11})',
                url
            )
    if match:
        return match.group(1)
    return None

In [11]:
def get_processed_ids(csv_path):
    """Carrega os IDs já processados do CSV"""
    if not os.path.exists(csv_path):
        return set()
    df_existing = pd.read_csv(csv_path)
    if 'video_id' in df_existing.columns:
        return set(df_existing['video_id'].dropna())
    return set()

In [14]:
# recebe o json de todos links do youtube e retorna o id e o tipo do conteúdo
all_ids = []
errors = []

with open(file_path, "r", encoding="utf-8") as f:
    for line in f:
        data = json.loads(line.strip())
        url = data["url"]
        if not url:
            errors.append({"url":None,"motivo": "nenhuma URL"})
            continue
            
        video_id = get_ID(url)
        if not video_id:
            errors.append({"url": url, "motivo": "falha regex"})
        else:
            all_ids.append(video_id)


In [15]:
import json
import csv

with open("falhas_ids.csv", "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=["url", "motivo"])
    writer.writeheader()
    writer.writerows(errors)

print(f"Total de IDs extraídos: {len(all_ids)}")
print(f"Total de links que falharam: {len(errors)} (salvo em falhas_ids.csv)")

Total de IDs extraídos: 1911054
Total de links que falharam: 9738 (salvo em falhas_ids.csv)


#### São links quebrados que não levam à um vídeos - estes links foram salvos em falhas_ids.csv caso queira conferir

In [9]:
video_ids = set(all_ids)
print(f"Total de IDs únicos: {len(video_ids)}")

Total de IDs únicos: 1397385


#### De 1911054 links diferentes, todos levavam para 1397385 IDS, ou seja, links diferentes que levavam para o mesmo vídeo

In [10]:
# Suas chaves
api_keys = [
    "Chave API 1",
    "Chave API 2"
]

processed_ids = get_processed_ids(output_csv_path)
ids_to_process = [vid_id for vid_id in video_ids if vid_id not in processed_ids]

print(f"Encontrados {len(video_ids)} IDs. Já processados: {len(processed_ids)}. Restam: {len(ids_to_process)}")

Encontrados 1397385 IDs. Já processados: 1183727. Restam: 213679


In [11]:
if not ids_to_process:
    print("Nenhum vídeo novo para processar.")
else:
    current_key_index = 0
    youtube = build("youtube", "v3", developerKey=api_keys[current_key_index])
    print(f"Iniciando com a chave API #{current_key_index + 1}")

    results = []
    # Processa em lotes de 50 (limite da API)
    for i in range(0, len(ids_to_process), 50):
        batch_ids = ids_to_process[i:i + 50]

        try:
            request = youtube.videos().list(
                part="snippet,statistics",
                id=",".join(batch_ids)
            )
            response = request.execute()

            for item in response.get("items", []):
                stats = item.get("statistics", {})
                results.append({
                    "video_id": item["id"],
                    "title": item["snippet"]["title"],
                    "description": item["snippet"]["description"],
                    "channel_title": item["snippet"]["channelTitle"],
                    "published_at": item["snippet"]["publishedAt"],
                    "youtube_video_link": f'https://www.youtube.com/watch?v={item["id"]}',
                    "view_count": stats.get("viewCount", ""),
                    "like_count": stats.get("likeCount", ""),
                    "comment_count": stats.get("commentCount", "")
                })

        except HttpError as e:
            if e.resp.status == 403:
                print(f"Quota da chave #{current_key_index + 1} esgotada.")
                current_key_index += 1

                if current_key_index < len(api_keys):
                    print(f"TROCANDO para a chave API #{current_key_index + 1}.")
                    youtube = build("youtube", "v3", developerKey=api_keys[current_key_index])
                else:
                    print("Todas as chaves de API foram esgotadas. Parando o processo.")
                    break
            else:
                print(f"Ocorreu um erro na API: {e}")

        print(f"Progresso: {len(processed_ids) + len(results)} / {len(video_ids)} vídeos")

    if results:
        df_new = pd.DataFrame(results, columns=[
            "video_id",
            "title",
            "description",
            "channel_title",
            "published_at",
            "youtube_video_link",
            "view_count",
            "like_count",
            "comment_count"
        ])
        if os.path.exists(output_csv_path):
            df_new.to_csv(output_csv_path, mode='a', header=False, index=False, encoding='utf-8')
        else:
            df_new.to_csv(output_csv_path, index=False, encoding='utf-8')
        print(f"Processamento concluído. {len(results)} novos vídeos salvos em '{output_csv_path}'.")
    else:
        print("Nenhum resultado novo foi gerado.")

Iniciando com a chave API #1
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos
Progresso: 1183727 / 1397385 vídeos

#### Acredito que não extaiu todos por alguns vídeos ter sido excluídos, ou coisas nesse sentido

In [12]:
pd.set_option('display.max_colwidth', None)
df_final = pd.read_csv('youtube_titulos_saida.csv')

print(f"Dimensões do DataFrame: {df_final.shape}")

df_final.head().style.set_properties(**{'text-align': 'left'}).set_table_styles([dict(selector='th', props=[('text-align', 'left')])])

Dimensões do DataFrame: (1183816, 9)


,video_id,title,description,channel_title,published_at,youtube_video_link,view_count,like_count,comment_count
0,jEKzQV5oajY,Free bus travel for migrants scrapped. For 5 minutes,nan,Joe Marsh,2024-04-10T15:05:37Z,https://www.youtube.com/watch?v=jEKzQV5oajY,1057.000000,152.000000,32.000000
1,xrGGce8cmx8,What is Spiritual Warfare?,"This charge I commit unto thee, son Timothy, according to the prophecies which went before on thee, that thou by them mightest war a good warfare; 1 Tim 1:18",WWURD,2023-10-18T14:39:18Z,https://www.youtube.com/watch?v=xrGGce8cmx8,65.000000,6.000000,1.000000
2,uaozGpSc4nc,80 Putins Have Layers,"In February 2024, Tucker Carlson stood in front of the onion domes of St. Basil’s Cathedral in Moscow and announced his interview with Russian President Vladimir Putin. Some call Putin a brutal dictator with plans for military expansion after he invaded Ukraine. Some call Carlson a traitor for meeting with him. The two-hour long interview that aired on X included many questions and many layers of answers from ancient history to 21st century diplomacy. The format of the interview itself held many more unspoken layers. Is Putin an ogre? Or an onion? Both have layers! Take a deep dive with the ladies of the Ark as we attempt to peel them away to reveal the backstory within. —Streamed February 22, 2024 #putininterview #tuckercarlson #russia Join us for our livestreams weekly at Unauthorized.tv (https://unauthorized.tv) and on Telegram (https://t.me/fencingbearatprayer). Subscribe to Logos & Tolkien on Unauthorized.tv to join the live chat, and get all our content, including “The Forge of Tolkien” and “Medieval History.” Schedule and episode guide: https://www.dragoncommonroom.com/mosaic-ark. Podcast available on Apple, Spotify, iHeart, Pandora, and Amazon Music. Buy our books: https://www.dragoncommonroom.com/shop-1 Join our list: https://www.dragoncommonroom.com/contact About us: https://www.dragoncommonroom.com/about",The Mosaic Ark,2024-02-22T07:30:05Z,https://www.youtube.com/watch?v=uaozGpSc4nc,330.000000,8.000000,6.000000
3,TwACgO_oPq4,Anacondaz — Акуле плевать (Official Music Video),#Anacondaz #Акулеплевать #Безпаники Long Story production. http://longstory.ru Слушать альбом «Перезвони мне +79995771202» https://orcd.co/perezvonimne Мерч и афиша концертов на сайте anacondaz.ru Anacondaz в соцсетях: http://vk.com/anacondaz https://fb.com/rapanacondaz https://t.me/anacondaz_official http://youtube.com/rapanacondaz http://twitter.com/rap_anacondaz http://instagram.com/rap_anacondaz https://www.tiktok.com/@rap_anacondaz По всем вопросам: +74952129294 event@anacondaz.ru #anacondaz #акулеплевать #dontpanic,ANACONDAZ,2014-05-20T09:29:45Z,https://www.youtube.com/watch?v=TwACgO_oPq4,2321870.000000,24565.000000,561.000000
4,A59ftbhsQUE,𝐆𝐀𝐋𝐀𝐂𝐓𝐈𝐂 𝐀𝐋𝐋𝐈𝐀𝐍𝐂𝐄 𝐌𝐄𝐒𝐒𝐀𝐆𝐄|𝐁𝐈𝐆 𝐍𝐄𝐖𝐒!~𝐑𝐄𝐌𝐀𝐈𝐍 𝐀𝐋𝐄𝐑𝐓 !~𝐏𝐑𝐀𝐘 𝐅𝐎𝐑 𝐇𝐔𝐌𝐀𝐍𝐈𝐓𝐘,#pleiadians #5dimension #ashtar #pleiadians #5dimension #ashtar #pleiadians #5dimension #ashtar #ashtar #ashtarsheran #pleiadians #pleiadianos #sanandreas #starseed #endtimemadness #5d #3danimation #3dimension #5dimension #starwars,GALACTIC ALLIANCE,2024-10-27T04:30:04Z,https://www.youtube.com/watch?v=A59ftbhsQUE,806.000000,136.000000,18.000000


In [ ]:
# Célula pra testar se a API Key ta funcionando
CHAVE_API_PARA_TESTAR = "COLE_AQUI_SUA_CHAVE_MAIS_NOVA"

print(f"Tentando usar a chave que termina em: '...{CHAVE_API_PARA_TESTAR[-4:]}'")

try:
    youtube = build('youtube', 'v3', developerKey=CHAVE_API_PARA_TESTAR)
    
    request = youtube.videos().list(
        part="snippet",
        id="dQw4w9WgXcQ" 
    )
    response = request.execute()

    print("\n✅ SUCESSO! A chave de API está funcionando corretamente.")
    print(f"Título do vídeo encontrado: {response['items'][0]['snippet']['title']}")

except HttpError as e:
    print("\n❌ FALHA! A chave não funcionou.")
    print("--- MENSAGEM DE ERRO DETALHADA ---")
    print(e)
    print("------------------------------------")

except Exception as e:
    print(f"\n❌ FALHA! Ocorreu um erro inesperado: {e}")